In [99]:
%reset -f
%matplotlib qt5
import numpy as np
import matplotlib.pyplot as plt
plt.close("all")

In [100]:
# Q.1
Fe = 4       # 4Hz
Te = 1/Fe    # Période d'échantillonnage
D = 10       # Synthèse sur 10s 
N = D*Fe + 1 # D = (N-1)*Te => N = D*Fe + 1
t = np.arange(0, N*Te, Te)
e = 3 + np.sin(2*np.pi*t)
plt.subplot(3,1,1)
plt.plot(t, e)
plt.title("Exercice IV")
plt.xlabel("nTe (s)")
plt.ylabel("e(nTe)");
figManager = plt.get_current_fig_manager()
figManager.window.showMaximized()

In [101]:
# Q.2
import scipy.signal as signal
help(signal.lfilter)

Help on function lfilter in module scipy.signal._signaltools:

lfilter(b, a, x, axis=-1, zi=None)
    Filter data along one-dimension with an IIR or FIR filter.

    Filter a data sequence, `x`, using a digital filter.  This works for many
    fundamental data types (including Object type).  The filter is a direct
    form II transposed implementation of the standard difference equation
    (see Notes).

    The function `sosfilt` (and filter design using ``output='sos'``) should be
    preferred over `lfilter` for most filtering tasks, as second-order sections
    have fewer numerical problems.

    Parameters
    ----------
    b : array_like
        The numerator coefficient vector in a 1-D sequence.
    a : array_like
        The denominator coefficient vector in a 1-D sequence.  If ``a[0]``
        is not 1, then both `a` and `b` are normalized by ``a[0]``.
    x : array_like
        An N-dimensional input array.
    axis : int, optional
        The axis of the input data array al

In [102]:
# Q.3 - Filtre Moyenneur sur quatre points
b = 1/4*np.ones(4)
a = [1];
s_moyenneur = signal.lfilter(b, a, e)
plt.subplot(3,1,2)
plt.plot(t, s_moyenneur)
plt.xlabel("nTe (s)")
plt.ylabel("s_moyenneur(nTe)");

In [103]:
# Q.4 - Filtre dérivateur ordre un
b = np.array([1/Te, -1/Te])
a = [1]
s_derivateur = signal.lfilter(b, a, e)

In [104]:
plt.subplot(3,1,3)
plt.plot(t, s_derivateur, "ko")
plt.xlabel("nTe (s)")
plt.ylabel("s_derivateur(nTe)");

Q.5 L'entrée du dérivateur a pour expression en discret (avec Fe=4Hz) :
$$e(nT_e)=3+sin(2\pi nT_e)$$
Le dérivateur utilise l'équation aux différences suivante :
$$s(nT_e)=\frac{e(nT_e)-e[(n-1)T_e]}{T_e}$$
Dont la transformée en z est :
$$S(z) = \frac{E(z)-z^{-1}E(z)}{T_e}$$
C.a.d. que :
$$
H(z)=\frac{S(z)}{E(z)}=\frac{1-z^{-1}}{T_e}=\frac{z^{\frac{1}{2}}-z^{-\frac{1}{2}}}{T_e}z^{-\frac{1}{2}}
$$
En posant :
$$z=e^{2i\pi\frac{k}{N}}=e^{2i\pi\frac{f}{F_e}}\text{ avec }f\in[0,\frac{F_e}{2}]$$
On obtient :
$$H(f)=2F_esin(\frac{\pi f}{F_e})e^{-i(\frac{\pi f}{F_e}-\frac{\pi}{2})}$$
Le signal d'entrée est composé de deux ondes aux fréquences 0Hz et 1Hz donc:
$$H(0) = 0 \text{ et } H(1) = 2F_esin(\frac{\pi}{4})e^{\frac{i\pi}{4}}=\sqrt{2}F_ee^{i\frac{\pi}{4}}$$
C.a.d que l'onde de fréquence 0Hz est totalement supprimée par le filtre et que l'onde de 1Hz voit sont amplitude multipliée par : $$\sqrt{2}F_e$$ et sa phase décalée de : $$\frac{\pi}{4}$$
Le signal de sortie peut alors s'écrire :
$$s(nT_e) = F_e\sqrt{2}sin(2\pi nT_e+\frac{\pi}{4})$$
La dérivée théorique vaut :
$$ 2\pi cos(2\pi nT_e)$$

In [105]:
# Q.5 - Calcul dérivée théorique
derivee_theorique = 2*np.pi*np.cos(2*np.pi*t)
s_derivateur_calcule = Fe*np.sqrt(2)*np.sin(2*np.pi*t+np.pi/4)
plt.plot(t, derivee_theorique, "ro")
plt.plot(t, s_derivateur_calcule, "bo")
plt.ylabel("")
plt.legend(["s_derivateur", "derivee_theorique", "s_derivateur_calcule"]);

In [106]:
plt.tight_layout()

On constate une légère différence entre le dérivateur et le dérivateur calculé. Cette différence vient des conditions initiales lors de l'application de la fonction signal.lfilter() : que valent l'entrée et la sortie à l'instant initial ? 
Si : $$nTe = 0\text{ c.a.d. si }n=0$$
on a alors : 
$$s(0) = \frac{e(0) - e(-T_e)}{T_e}$$
Il nous faut donc connaitre la valeur de l'entrée à l'instant -Te :
$$e(-T_e) = e[(-1)T_e]=3+sin[2\pi (-1)Te]=3+sin(-\frac{\pi}{2}) = 3-1=2$$
Dans ce cas :
$$s(0) = \frac{3 - 2}{T_e} = 4$$
Les conditions initiales a utiliser sont donc une valeur de 4 pour la sortie à l'instant 0 et une valeur de 2 pour l'entrée à l'instant -Te. On pourra fournir ces valeurs à la fonction signal.lfiltic pour utiliser ce resultat dans la fonction lfilter comme ci-dessous.

In [108]:
# Q.5 - suite
b = np.array([1/Te, -1/Te])
a = [1]
zi = signal.lfiltic(b, a, [4], [2])
s_derivateur_ic = signal.lfilter(b, a, e, zi = zi)[0]
plt.plot(t, s_derivateur_ic, "go")
plt.legend(["s_derivateur", "derivee_theorique", "s_derivateur_calcule", "s_derivateur_ic"]);

On constate alors que le dérivateur calculé théoriquement et le dérivateur calculé avec conditions initiales coïncident. On peut également constater que le dérivateur calculé d'ordre 1 n'est qu'une approximation de la dérivée théorique.